In [7]:
# Imports

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.tuner import Tuner
from ray import tune
from ray.tune.integration.pytorch_lightning import TuneReportCallback
import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## Getting Data

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadmusharraf444/used-car-price-prediction-dataset")
data_path = os.path.join(path, "quikr_car.csv")

print("Path to dataset files:", data_path)

Path to dataset files: /home/franio/.cache/kagglehub/datasets/muhammadmusharraf444/used-car-price-prediction-dataset/versions/1/quikr_car.csv


In [36]:
data_df = pd.read_csv(data_path)

data_df.dropna(inplace=True)
data_df = data_df[data_df.Price != 'Ask For Price' ]
data_df.Price = data_df.Price.str.replace(',', '').astype(float)

data_df.kms_driven = data_df.kms_driven.str.strip('kms').str.replace(',', '', regex=False).str.strip().astype(float)
data_df.year = data_df.year.astype(int)
data_df = pd.get_dummies(data_df, columns=['company', 'fuel_type'], dtype=int)

data_df.drop(['name'], axis=1, inplace=True)

data_df

,year,Price,kms_driven,company_Audi,company_BMW,company_Chevrolet,company_Datsun,company_Fiat,company_Force,company_Ford,...,company_Nissan,company_Renault,company_Skoda,company_Tata,company_Toyota,company_Volkswagen,company_Volvo,fuel_type_Diesel,fuel_type_LPG,fuel_type_Petrol
0,2007,80000.0,45000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,2006,425000.0,40.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,2014,325000.0,28000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,2014,575000.0,36000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
6,2012,175000.0,41000.0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,2011,270000.0,50000.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
885,2009,110000.0,30000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
886,2009,300000.0,132000.0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
888,2018,260000.0,27000.0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0


In [44]:
X,y = data_df.drop('Price', axis=1).to_numpy(), data_df['Price'].to_numpy()

X.dtype

dtype('float64')

# Classes

In [45]:
class MyDataset(Dataset):
 def __init__(self, data, outputs):
  """Inicjalizacja - przygotowanie struktury danych"""
  self.data = data
  self.outputs = outputs

 def __len__(self):
  """Zwraca liczbę próbek w datasecie"""
  return len(self.data)

 def __getitem__(self, idx):
  """Zwraca próbkę o indeksie idx"""
  x = self.data[idx]
  y = self.outputs[idx]
  return x, y

class MyDataModule(pl.LightningDataModule):
 def __init__(self,X:np.ndarray, y:float, batch_size=32):
  super().__init__()
  X_tensor = torch.from_numpy(X).float()
  y_tensor = torch.from_numpy(y).float()

  self.X = X_tensor
  self.y = y_tensor
  self.batch_size = batch_size


 def setup(self, stage=None):
  X_train_val, X_test, y_train_val, y_test = train_test_split(self.X, self.y, test_size=0.1)
  X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2)
  self.train_dataset = MyDataset(X_train, y_train)
  self.val_dataset = MyDataset(X_val, y_val)
  self.test_dataset = MyDataset(X_test, y_test)

 def train_dataloader(self):
  return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

 def val_dataloader(self):
  return DataLoader(self.val_dataset, batch_size=self.batch_size)

 def test_dataloader(self):
  return DataLoader(self.test_dataset, batch_size=self.batch_size)

In [46]:
class LitModel(pl.LightningModule):
 def __init__(self, input_size, hidden_size, output_dim,lr=0.001):
  super().__init__()
  self.save_hyperparameters() # Call this first to save __init__ arguments

  # to self.hparams
  self.layer1 = nn.Linear(self.hparams.input_size, self.hparams.hidden_size)
  self.layer2 = nn.Linear(self.hparams.hidden_size, self.hparams.hidden_size)
  self.layer3 = nn.Linear(self.hparams.hidden_size, self.hparams.output_dim)
  self.criterion = nn.MSELoss()

 def forward(self, x):
  x = torch.relu(self.layer1(x))
  x = torch.relu(self.layer2(x))
  x = self.layer3(x)
  return x

 def training_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  self.log('train_loss', loss)
  return loss

 def validation_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('val_loss', loss, prog_bar=True)
  self.log('val_r2', r2, prog_bar=True)

 def test_step(self, batch, batch_idx):
  x, y = batch
  y_hat = self(x).squeeze()
  loss = self.criterion(y_hat, y)
  r2 = r2_score(y.detach().cpu().numpy().squeeze(), y_hat.detach().cpu().numpy().squeeze())
  self.log('test_loss', loss)
  self.log('test_r2', r2)

 def configure_optimizers(self):
  # Use the learning rate from self.hparams
  return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)